# Case-Level Feature Attribution vs Frequency

Does the model attribute more to a case-level categorical feature (e.g. `case:LoanGoal`)
when that feature value was common in training?

**Note:** BPIC17 does not have a `Variant index` feature (unlike Helpdesk).
We use `case:LoanGoal` as the case-level categorical feature instead.

In [ ]:
import sys
from pathlib import Path
_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir(): break
    _current = _current.parent
sys.path.insert(0, str(_current))
sys.path.insert(0, str(_current / 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter
from tqdm.auto import tqdm

from src.interpretability.config.bpic17_config import CONFIG
CONFIG.use_improved = False  # set True for the improved variant
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM
from src.evaluation.evaluation import Evaluation
from src.interpretability import InterpretabilityTool

%matplotlib inline

# Load
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()
test_dataset = torch.load(str(CONFIG.get_test_data_path()), weights_only=False)
eval_helper = Evaluation(model=model, dataset=test_dataset, concept_name=CONFIG.concept_name,
                         growing_num_values=CONFIG.growing_num_values, all_cat=CONFIG.all_cat, all_num=CONFIG.all_num)
tool = InterpretabilityTool(model, model.data_set_categories, device='cpu')

# Find case:LoanGoal position among categorical features.
# BPIC17 does NOT have 'Variant index'; we use 'case:LoanGoal' as the
# case-level categorical feature for this frequency-attribution analysis.
CASE_FEATURE = 'case:LoanGoal'

try:
    feat_idx_pos = next(i for i, (n, _, _) in enumerate(model.data_set_categories[0])
                        if n == CASE_FEATURE)
except StopIteration:
    raise RuntimeError(
        f"Feature '{CASE_FEATURE}' not found in model.data_set_categories[0]. "
        f"Available: {[n for n, _, _ in model.data_set_categories[0]]}"
    )

feat_name = model.data_set_categories[0][feat_idx_pos][0]
feat_mapping = model.data_set_categories[0][feat_idx_pos][2]  # {label: index}
inv_mapping = {v: k for k, v in feat_mapping.items()}  # {index: label}
print(f"Cases: {len(eval_helper.cases)}, Feature: '{feat_name}' at pos {feat_idx_pos}")
print(f"Feature values ({len(feat_mapping)}): {list(feat_mapping.keys())}")

Data set categories:  ([('concept:name', 28, {'A_Accepted': 1, 'A_Cancelled': 2, 'A_Complete': 3, 'A_Concept': 4, 'A_Create Application': 5, 'A_Denied': 6, 'A_Incomplete': 7, 'A_Pending': 8, 'A_Submitted': 9, 'A_Validating': 10, 'EOS': 11, 'O_Accepted': 12, 'O_Cancelled': 13, 'O_Create Offer': 14, 'O_Created': 15, 'O_Refused': 16, 'O_Returned': 17, 'O_Sent (mail and online)': 18, 'O_Sent (online only)': 19, 'W_Assess potential fraud': 20, 'W_Call after offers': 21, 'W_Call incomplete files': 22, 'W_Complete application': 23, 'W_Handle leads': 24, 'W_Personal Loan collection': 25, 'W_Shortened completion ': 26, 'W_Validate application': 27}), ('Action', 7, {'Created': 1, 'Deleted': 2, 'EOS': 3, 'Obtained': 4, 'Released': 5, 'statechange': 6}), ('org:resource', 151, {'EOS': 1, 'User_1': 2, 'User_10': 3, 'User_100': 4, 'User_101': 5, 'User_102': 6, 'User_103': 7, 'User_104': 8, 'User_105': 9, 'User_106': 10, 'User_107': 11, 'User_108': 12, 'User_109': 13, 'User_11': 14, 'User_110': 15, 'U

In [ ]:
# Count feature-value frequencies per case
def get_feature_value(case_name):
    """Return the encoded index of the case-level feature for this case."""
    case = eval_helper.cases[case_name]
    for _, prefix, _ in eval_helper._iterate_case(case):
        t = prefix[0][feat_idx_pos].squeeze()
        for v in t.tolist():
            if v != 0:
                return v
        break
    return 0

case_feat_values = {c: get_feature_value(c) for c in eval_helper.cases}
feat_value_counts = Counter(case_feat_values.values())

# Show decoded labels
print(f"Unique {feat_name} values: {len(feat_value_counts)}")
for val, cnt in feat_value_counts.most_common():
    label = inv_mapping.get(val, f'<idx {val}>')
    print(f"  {label}: {cnt} cases")

In [ ]:
# Sample cases (larger sample)
N_SAMPLES = 60
PREFIX_LENGTH = 3

np.random.seed(42)
all_cases = list(case_feat_values.keys())
sampled = np.random.choice(all_cases, min(N_SAMPLES, len(all_cases)), replace=False)
print(f"Sampled {len(sampled)} cases")

In [ ]:
# Compute attributions
results = []

for case_name in tqdm(sampled, desc="Computing"):
    case = eval_helper.cases[case_name]
    prefix = None
    for pl, p, _ in eval_helper._iterate_case(case):
        if pl >= PREFIX_LENGTH:
            prefix = p
            break
    if prefix is None: continue
    
    try:
        process = ([t.squeeze(0) for t in prefix[0]], [t.squeeze(0) for t in prefix[1]])
        attr_map = tool.compute_attribution_map(
            process=process, prefix_length=PREFIX_LENGTH, target=CONFIG.concept_name,
            target_class='auto', suffix_scope='step', suffix_step=0,
            method='integrated_gradients', n_steps=CONFIG.ig_steps
        )
        
        feat_attr = attr_map.attributions[feat_name]
        if hasattr(feat_attr, 'detach'): feat_attr = feat_attr.detach().cpu().numpy()
        feat_mag = np.abs(feat_attr).sum()
        
        total_mag = sum(np.abs(v.detach().cpu().numpy() if hasattr(v, 'detach') else v).sum() 
                       for v in attr_map.attributions.values())
        
        feat_val = case_feat_values[case_name]
        results.append({
            'feat_count': feat_value_counts[feat_val],
            'feat_label': inv_mapping.get(feat_val, f'<idx {feat_val}>'),
            'feat_attribution': feat_mag,
            'feat_ratio': feat_mag / total_mag if total_mag > 0 else 0
        })
    except: pass

print(f"Computed: {len(results)}")

In [ ]:
# Analysis
freqs = np.array([r['feat_count'] for r in results])
attrs = np.array([r['feat_attribution'] for r in results])
ratios = np.array([r['feat_ratio'] for r in results])

corr, p_val = stats.pearsonr(freqs, attrs)
corr_log, p_log = stats.pearsonr(np.log1p(freqs), attrs)
corr_ratio, p_ratio = stats.pearsonr(freqs, ratios)

print(f"Feature: {feat_name}")
print(f"Correlation (freq vs attribution):     r={corr:.4f}, p={p_val:.4f} {'*' if p_val<0.05 else ''}")
print(f"Correlation (log freq vs attribution): r={corr_log:.4f}, p={p_log:.4f} {'*' if p_log<0.05 else ''}")
print(f"Correlation (freq vs ratio):           r={corr_ratio:.4f}, p={p_ratio:.4f} {'*' if p_ratio<0.05 else ''}")

# Per-value breakdown
print(f"\nPer-value mean attribution:")
from collections import defaultdict
by_label = defaultdict(list)
for r in results:
    by_label[r['feat_label']].append(r['feat_attribution'])
for label in sorted(by_label, key=lambda l: -np.mean(by_label[l])):
    vals = by_label[label]
    count = feat_value_counts.get(
        next((k for k, v in inv_mapping.items() if v == label), None), 0)
    print(f"  {label} (n={len(vals)}, freq={count}): "
          f"mean_attr={np.mean(vals):.4f}, std={np.std(vals):.4f}")

In [ ]:
# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(freqs, attrs, alpha=0.5)
z = np.polyfit(freqs, attrs, 1); axes[0].plot(np.sort(freqs), np.poly1d(z)(np.sort(freqs)), 'r--')
axes[0].set_xlabel(f'{feat_name} Frequency'); axes[0].set_ylabel(f'{feat_name} Attribution')
axes[0].set_title(f'Frequency vs Attribution (r={corr:.3f})')

axes[1].scatter(np.log1p(freqs), attrs, alpha=0.5)
z = np.polyfit(np.log1p(freqs), attrs, 1); axes[1].plot(np.sort(np.log1p(freqs)), np.poly1d(z)(np.sort(np.log1p(freqs))), 'r--')
axes[1].set_xlabel('Log(Frequency)'); axes[1].set_ylabel(f'{feat_name} Attribution')
axes[1].set_title(f'Log Freq vs Attribution (r={corr_log:.3f})')

axes[2].scatter(freqs, ratios, alpha=0.5)
z = np.polyfit(freqs, ratios, 1); axes[2].plot(np.sort(freqs), np.poly1d(z)(np.sort(freqs)), 'r--')
axes[2].set_xlabel(f'{feat_name} Frequency'); axes[2].set_ylabel('Attribution Ratio')
axes[2].set_title(f'Frequency vs Ratio (r={corr_ratio:.3f})')

plt.suptitle(f'BPIC17: {feat_name} Attribution vs Frequency', y=1.02)
plt.tight_layout()
plt.show()

# Conclusion
print("\n" + "="*50)
if p_val < 0.05:
    direction = "MORE" if corr > 0 else "LESS"
    print(f"SIGNIFICANT: Model attributes {direction} to {feat_name} for common values")
else:
    print(f"NOT SIGNIFICANT: No clear relationship between {feat_name} frequency and attribution")